### Silver to gold - Building BI ready tables

In [0]:
df_products = spark.table("ecommerce.silver.slv_products")
df_brands = spark.table("ecommerce.silver.silver_brands")
df_category = spark.table("ecommerce.silver.slv_category")

In [0]:
df_products.createOrReplaceTempView("v_products")
df_brands.createOrReplaceTempView("v_brands")
df_category.createOrReplaceTempView("v_category")

In [0]:
display(spark.sql("Select * from v_products limit 5"))

product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp
2000000000015,STCR-HNK-00001,HNK,STCR,White,One-Size,Coton,305,22.2,17.1,6.3,0,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-09-04T04:36:53.190Z
2000000000022,HMNS-HNK-00002,HNK,HMNS,Silver,One-Size,Steel,682,18.2,12.3,3.7,1,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-09-04T04:36:53.190Z
2000000000039,NOVW-CE-00003,CE,NOVW,Purple,One-Size,Wood,243,18.2,13.9,4.2,0,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-09-04T04:36:53.190Z
2000000000046,URTL-APP-00004,APP,URTL,Silver,S,Ruber,225,17.6,4.6,5.8,50,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-09-04T04:36:53.190Z
2000000000053,GGRN-GRC-00005,GRCY,GGRN,Silver,One-Size,Ruber,455,27.2,15.8,7.4,4,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-09-04T04:36:53.190Z


In [0]:
display(spark.sql("Select * from v_category limit 5"))

category_code,category_name,_ingested_at,_source_file
BPC,Beauty & Personal Care,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
SPT,Sports & Outdoors,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
APP,Apparel,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
GRCY,Grocery,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
HNK,Home & Kitchen,2026-09-03T16:57:22.497Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv


In [0]:
display(spark.sql("Select * from v_brands limit 5"))

brand_code,brand_name,Category_code,_source_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-09-03T16:45:32.768Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-09-03T16:45:32.768Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-09-03T16:45:32.768Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-09-03T16:45:32.768Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-09-03T16:45:32.768Z


In [0]:
%sql
-- build brands and category mapping and write gold tsqlable

create or replace table ecommerce.gold.gld_dim_products as 

with brands_Categories AS (
Select 
    b.brand_code,
    b.brand_name,
    c.category_code,
    c.category_name
from v_brands b
inner join v_category c
on b.Category_code = c.category_code
)

Select 
    p.product_id,
    p.sku,
    bc.brand_code,
    bc.brand_name,
    bc.category_code,
    bc.category_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    p.rating_count
from v_products p
left join brands_Categories bc
on p.brand_code = bc.brand_code


num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verify the gold dimension table
SELECT 
    product_id,
    sku,
    brand_name,
    category_name,
    color,
    size,
    material,
    rating_count
FROM ecommerce.gold.gld_dim_products
LIMIT 20

product_id,sku,brand_name,category_name,color,size,material,rating_count
2000000000015,STCR-HNK-00001,SteelCraft,Home & Kitchen,White,One-Size,Coton,0
2000000000022,HMNS-HNK-00002,HomeNest,Home & Kitchen,Silver,One-Size,Steel,1
2000000000039,NOVW-CE-00003,NovaWave,Electronics,Purple,One-Size,Wood,0
2000000000046,URTL-APP-00004,UrbanTrail,Apparel,Silver,S,Ruber,50
2000000000053,GGRN-GRC-00005,GoodGrain,Grocery,Silver,One-Size,Ruber,4
2000000000060,SLKE-BPC-00006,SilkEssence,Beauty & Personal Care,Purple,One-Size,Plastic,0
2000000000077,VOLT-CE-00007,VoltEdge,Electronics,Blue,One-Size,Plastic,5
2000000000084,CBLT-APP-00008,CobaltWear,Apparel,Blue,XS,Polyester,0
2000000000091,ARFT-SPT-00009,AeroFit,Sports & Outdoors,Blue,XL,Plastic,11
2000000000107,MOSA-APP-0000A,Mosaic,Apparel,White,L,Polyester,6


### Customers

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  


In [0]:
country_state_map

{'India': {'MH': 'West',
  'GJ': 'West',
  'RJ': 'West',
  'KA': 'South',
  'TN': 'South',
  'TS': 'South',
  'AP': 'South',
  'KL': 'South',
  'UP': 'North',
  'WB': 'North',
  'DL': 'North'},
 'Australia': {'VIC': 'SouthEast',
  'WA': 'West',
  'NSW': 'East',
  'QLD': 'NorthEast'},
 'United Kingdom': {'ENG': 'England',
  'WLS': 'Wales',
  'NIR': 'Northern Ireland',
  'SCT': 'Scotland'},
 'United States': {'MA': 'NorthEast',
  'FL': 'South',
  'NJ': 'NorthEast',
  'CA': 'West',
  'NY': 'NorthEast',
  'TX': 'South'},
 'United Arab Emirates': {'AUH': 'Abu Dhabi', 'DU': 'Dubai', 'SHJ': 'Sharjah'},
 'Singapore': {'SG': 'Singapore'},
 'Canada': {'BC': 'West',
  'AB': 'West',
  'ON': 'East',
  'QC': 'East',
  'NS': 'East',
  'IL': 'Other'}}

In [0]:
# 1 Flatten country_state_map into a list of Rows
from pyspark.sql import Row

rows = []
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))
rows[:10]     

[Row(country='India', state='MH', region='West'),
 Row(country='India', state='GJ', region='West'),
 Row(country='India', state='RJ', region='West'),
 Row(country='India', state='KA', region='South'),
 Row(country='India', state='TN', region='South'),
 Row(country='India', state='TS', region='South'),
 Row(country='India', state='AP', region='South'),
 Row(country='India', state='KL', region='South'),
 Row(country='India', state='UP', region='North'),
 Row(country='India', state='WB', region='North')]

In [0]:
df_region_mapping = spark.createDataFrame(rows)
df_region_mapping.show(truncate = False)

+--------------+-----+----------------+
|country       |state|region          |
+--------------+-----+----------------+
|India         |MH   |West            |
|India         |GJ   |West            |
|India         |RJ   |West            |
|India         |KA   |South           |
|India         |TN   |South           |
|India         |TS   |South           |
|India         |AP   |South           |
|India         |KL   |South           |
|India         |UP   |North           |
|India         |WB   |North           |
|India         |DL   |North           |
|Australia     |VIC  |SouthEast       |
|Australia     |WA   |West            |
|Australia     |NSW  |East            |
|Australia     |QLD  |NorthEast       |
|United Kingdom|ENG  |England         |
|United Kingdom|WLS  |Wales           |
|United Kingdom|NIR  |Northern Ireland|
|United Kingdom|SCT  |Scotland        |
|United States |MA   |NorthEast       |
+--------------+-----+----------------+
only showing top 20 rows


In [0]:
df_silver = spark.table("ecommerce.silver.slv_customers")
display(df_silver.limit(5))

customer_id,phone,country_code,country,state,file_name,ingest_timestamp
CUST000000000001,917280033536.0,IN,India,MH,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-09-04T04:52:29.509Z
CUST000000000002,619489725433.0,AU,Australia,VIC,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-09-04T04:52:29.509Z
CUST000000000003,919390066524.0,IN,India,TN,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-09-04T04:52:29.509Z
CUST000000000004,917073741793.0,IN,India,TN,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-09-04T04:52:29.509Z
CUST000000000005,618478772532.0,AU,Australia,WA,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-09-04T04:52:29.509Z


In [0]:
df_gold = df_silver.join(df_region_mapping, on =["country","state"], \
    how ="left"  )

df_gold= df_gold.fillna({"region": "Other"})

df_gold.show()

+--------------------+-----+----------------+---------------+------------+--------------------+--------------------+----------------+
|             country|state|     customer_id|          phone|country_code|           file_name|    ingest_timestamp|          region|
+--------------------+-----+----------------+---------------+------------+--------------------+--------------------+----------------+
|               India|   MH|CUST000000000001| 917280033536.0|          IN|dbfs:/Volumes/eco...|2026-09-04 04:52:...|            West|
|           Australia|  VIC|CUST000000000002| 619489725433.0|          AU|dbfs:/Volumes/eco...|2026-09-04 04:52:...|       SouthEast|
|               India|   TN|CUST000000000003| 919390066524.0|          IN|dbfs:/Volumes/eco...|2026-09-04 04:52:...|           South|
|               India|   TN|CUST000000000004| 917073741793.0|          IN|dbfs:/Volumes/eco...|2026-09-04 04:52:...|           South|
|           Australia|   WA|CUST000000000005| 618478772532.0| 

In [0]:
df_gold.write.format("delta").mode("overwrite")\
.option("mergeschema",True).saveAsTable("ecommerce.gold.gld_customers")    

### Date/Calendar

In [0]:
df_silver = spark.table("ecommerce.silver.slv_calendar")
display(df_silver.limit(5))

date,year,day_name,quarter,week,_ingested_at,_source_file
2025-08-06,2025,Wednesday,Q3-2025,Week32-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv
2025-08-16,2025,Saturday,Q3-2025,Week33-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv
2025-08-22,2025,Friday,Q3-2025,Week34-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv
2025-09-11,2025,Thursday,Q3-2025,Week37-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv
2025-09-20,2025,Saturday,Q3-2025,Week38-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv


In [0]:
from pyspark.sql import functions as F
df_gold = df_silver.withColumn("date_id", F.date_format(F.col("date"), "yyyyMMdd").cast("int"))

# Add month name (e.g., 'January', 'February', etc.)
df_gold = df_gold.withColumn("month_name", F.date_format(F.col("date"), "MMMM"))

# Add is_weekend column
df_gold = df_gold.withColumn(
    "is_weekend",
    F.when(F.col("day_name").isin("Saturday", "Sunday"), 1).otherwise(0)
)

display(df_gold.limit(5))


date,year,day_name,quarter,week,_ingested_at,_source_file,date_id,month_name,is_weekend
2025-08-06,2025,Wednesday,Q3-2025,Week32-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,20250806,August,0
2025-08-16,2025,Saturday,Q3-2025,Week33-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,20250816,August,1
2025-08-22,2025,Friday,Q3-2025,Week34-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,20250822,August,0
2025-09-11,2025,Thursday,Q3-2025,Week37-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,20250911,September,0
2025-09-20,2025,Saturday,Q3-2025,Week38-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,20250920,September,1


In [0]:
desired_columns_order = ["date_id", "date", "year", "month_name", "day_name", "is_weekend", "quarter", "week", "_ingested_at", "_source_file"]

df_gold = df_gold.select(desired_columns_order)

display(df_gold.limit(5))

date_id,date,year,month_name,day_name,is_weekend,quarter,week,_ingested_at,_source_file
20250806,2025-08-06,2025,August,Wednesday,0,Q3-2025,Week32-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv
20250816,2025-08-16,2025,August,Saturday,1,Q3-2025,Week33-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv
20250822,2025-08-22,2025,August,Friday,0,Q3-2025,Week34-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv
20250911,2025-09-11,2025,September,Thursday,0,Q3-2025,Week37-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv
20250920,2025-09-20,2025,September,Saturday,1,Q3-2025,Week38-2025,2026-09-04T04:57:14.816Z,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv


In [0]:
# write table to gold layer
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("ecommerce.gold.gld_dim_date")

In [0]:
%sql

DESCRIBE EXTENDED ecommerce.gold.gld_dim_date;

col_name,data_type,comment
date_id,int,null
date,date,null
year,int,null
month_name,string,null
day_name,string,null
is_weekend,int,null
quarter,string,null
week,string,null
_ingested_at,timestamp,null
_source_file,string,null
